# 문장 혐오 탐정기 + 정화된 문장 반환하기

In [ ]:
!pip install -q transformers datasets
!pip install -q sentencepiece
!pip install -q kobert-transformers
!pip install tf-keras

In [ ]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='klue/bert-base')

def fill_korean_mask(text, k=1):
    # print(f'{text =}')

    results = unmasker(text, top_k=k)
    # print(f'{results =}')
    
    # if list take first in list 
    top_result = results[0] if isinstance(results, list) else results
    
    return top_result['sequence']

/opt/homebrew/anaconda3/envs/nlp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at klue/bert-base were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [ ]:
# 혐오 찾기 LABEL_1=(O)혐오 | LABEL_0=(X)혐오
classifier = pipeline("text-classification", model="jinkyeongk/kcELECTRA-toxic-detector")

def moderator(text):
    result = classifier(text)[0]
    # print(f'{result = }')
    # 혐오 단어 masking 처리
    if result['label'] == 'LABEL_1' and result['score'] > 0.8:
        print("문장에서 혐오 표현이 감지되었습니다. 정화된 문장을 보여드릴게요.")

        bad_words = ["개새끼","어이없어", "진짜짜증나", "멘붕", "존나", "못생겼다"] 
        masked_text = text
        for word in bad_words:
            masked_text = masked_text.replace(word, "[MASK]")

        return(fill_korean_mask(masked_text))
    return text, "글렇군요!"


Device set to use mps:0


In [ ]:
# 실행하기 
while True:
    user_input = input('q 쓰면 나가기')
    if user_input=='q':
        break
    print(moderator(user_input))

result = {'label': 'LABEL_1', 'score': 0.9989683628082275}
text ='너 진짜 [MASK].'
results =[{'score': 0.6928194761276245, 'token': 18, 'token_str': '.', 'sequence': '너 진짜..'}]
너 진짜..
result = {'label': 'LABEL_1', 'score': 0.9977752566337585}
text ='너 진짜 [MASK] 알아?.'
results =[{'score': 0.09876959770917892, 'token': 16427, 'token_str': '누군지', 'sequence': '너 진짜 누군지 알아?.'}]
너 진짜 누군지 알아?.
result = {'label': 'LABEL_0', 'score': 0.997423529624939}
('ㅂ', '글렇군요!')
